# E033 — Adversarial Image Augmentation for Geometric Robustness

**Motivation:** E028 showed image system is brittle to geometric degradations (rotation ±15° → 7.3% EER, occlusion → 18%). ARoFace paper (2024) uses adversarial spatial transformations during training to improve robustness to face alignment errors.

**Hypothesis:** Training with adversarial geometric augmentations (small rotations, translations) will improve robustness to geometric degradations while maintaining clean performance.

**Approach:** Instead of random rotation, use adversarial training:
1. For each training image, find the rotation/translation that maximizes classification error
2. Add these "hard" examples to training
3. Test on clean + stressed data

**Configs:**
- `Baseline`: E007 +All (flip + brightness + noise)
- `+AdvRot`: + adversarial rotation ±10°
- `+AdvTrans`: + adversarial translation ±5px
- `+AdvBoth`: + both adversarial rotation + translation

In [1]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../src').resolve()))

import numpy as np
from PIL import Image
from scipy.ndimage import rotate, shift
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

SEED = 67
N_PCA = 50
C_LOGREG = 1.0

DATA = Path('../data').resolve()
manifest = load_manifest(DATA)
y_all = manifest['label'].to_numpy()
print(f'{len(manifest)} samples')

E007_REF = {'mean_eer': 0.97, 'std_eer': 0.86}

222 samples


## 1. Augmentation functions

In [2]:
def _load_image(path):
    img = Image.open(path).convert("RGB")
    if img.size != (80, 80):
        img = img.resize((80, 80), Image.BILINEAR)
    return np.array(img, dtype=np.float32).mean(axis=2)

def _aug_flip(x):
    return x[:, ::-1].copy()

def _aug_bright(x, rng):
    return np.clip(x * rng.uniform(0.7, 1.3), 0, 255)

def _aug_inoise(x, rng):
    return np.clip(x + rng.normal(0, 15, x.shape), 0, 255)

def _aug_rotate(x, angle, order=1):
    """Rotate image by angle degrees."""
    return rotate(x, angle, reshape=False, order=order, mode='constant', cval=0)

def _aug_translate(x, dx, dy, order=1):
    """Translate image by dx, dy pixels."""
    return shift(x, (dy, dx), order=order, mode='constant', cval=0)

def find_adversarial_rotation(x, scaler, pca, clf, rng, max_angle=10, n_steps=5):
    """Find rotation angle that maximizes classification error (minimizes correct class logit)."""
    angles = np.linspace(-max_angle, max_angle, n_steps)
    best_angle = 0
    worst_logit = np.inf
    
    x_flat = x.flatten().reshape(1, -1)
    x_pca = pca.transform(scaler.transform(x_flat))
    base_logit = clf.decision_function(x_pca)[0]
    
    for angle in angles:
        if abs(angle) < 0.1:
            continue
        x_rot = _aug_rotate(x, angle)
        x_rot_flat = x_rot.flatten().reshape(1, -1)
        x_rot_pca = pca.transform(scaler.transform(x_rot_flat))
        logit = clf.decision_function(x_rot_pca)[0]
        
        # For target samples, we want to minimize logit; for non-target, maximize
        # Here we just minimize absolute logit (make it closer to decision boundary)
        if abs(logit) < abs(worst_logit):
            worst_logit = logit
            best_angle = angle
    
    return best_angle

def find_adversarial_translation(x, scaler, pca, clf, rng, max_shift=5, n_steps=5):
    """Find translation that maximizes classification error."""
    shifts = np.linspace(-max_shift, max_shift, n_steps)
    best_dx, best_dy = 0, 0
    worst_logit = np.inf
    
    x_flat = x.flatten().reshape(1, -1)
    x_pca = pca.transform(scaler.transform(x_flat))
    
    for dx in shifts:
        for dy in shifts:
            if abs(dx) < 0.1 and abs(dy) < 0.1:
                continue
            x_trans = _aug_translate(x, dx, dy)
            x_trans_flat = x_trans.flatten().reshape(1, -1)
            x_trans_pca = pca.transform(scaler.transform(x_trans_flat))
            logit = clf.decision_function(x_trans_pca)[0]
            
            if abs(logit) < abs(worst_logit):
                worst_logit = logit
                best_dx, best_dy = dx, dy
    
    return best_dx, best_dy

print('Augmentation functions defined')

Augmentation functions defined


## 2. Training with adversarial augmentation

In [3]:
def _train_image_with_aug(df, data_dir, augment_type, seed):
    """Train image model with specified augmentation type."""
    rng = np.random.default_rng(seed)
    X, y = [], []
    
    # First pass: fit scaler and PCA on original + basic aug
    X_basic, y_basic = [], []
    for _, row in df.iterrows():
        orig = _load_image(_find_png(row["stem"], data_dir))
        vecs = [orig]
        if augment_type in ['+All', '+AdvRot', '+AdvTrans', '+AdvBoth']:
            vecs += [_aug_flip(orig), _aug_bright(orig, rng), _aug_inoise(orig, rng)]
        for v in vecs:
            X_basic.append(v.flatten()); y_basic.append(row["label"])
    
    X_basic = np.array(X_basic)
    scaler = StandardScaler()
    pca = PCA(n_components=N_PCA, random_state=SEED)
    X_pca = pca.fit_transform(scaler.fit_transform(X_basic))
    
    # Fit initial classifier
    clf = LogisticRegression(C=C_LOGREG, max_iter=1000, random_state=SEED)
    clf.fit(X_pca, y_basic)
    
    # Second pass: add adversarial examples
    X_adv, y_adv = [], []
    for _, row in df.iterrows():
        orig = _load_image(_find_png(row["stem"], data_dir))
        
        if augment_type == '+AdvRot':
            angle = find_adversarial_rotation(orig, scaler, pca, clf, rng)
            x_adv = _aug_rotate(orig, angle)
            X_adv.append(x_adv.flatten()); y_adv.append(row["label"])
        
        elif augment_type == '+AdvTrans':
            dx, dy = find_adversarial_translation(orig, scaler, pca, clf, rng)
            x_adv = _aug_translate(orig, dx, dy)
            X_adv.append(x_adv.flatten()); y_adv.append(row["label"])
        
        elif augment_type == '+AdvBoth':
            angle = find_adversarial_rotation(orig, scaler, pca, clf, rng)
            x_temp = _aug_rotate(orig, angle)
            dx, dy = find_adversarial_translation(x_temp, scaler, pca, clf, rng)
            x_adv = _aug_translate(x_temp, dx, dy)
            X_adv.append(x_adv.flatten()); y_adv.append(row["label"])
    
    # Combine basic + adversarial
    if len(X_adv) > 0:
        X_all = np.vstack([X_basic, X_adv])
        y_all = np.array(y_basic + y_adv)
    else:
        X_all = X_basic
        y_all = np.array(y_basic)
    
    # Refit scaler, pca, clf on all data
    scaler = StandardScaler()
    pca = PCA(n_components=N_PCA, random_state=SEED)
    clf = LogisticRegression(C=C_LOGREG, max_iter=1000, random_state=SEED)
    X_pca = pca.fit_transform(scaler.fit_transform(X_all))
    clf.fit(X_pca, y_all)
    
    return scaler, pca, clf

def _find_png(stem, data_dir):
    for sf in ("target_train", "target_dev", "non_target_train", "non_target_dev"):
        p = data_dir / sf / (stem + ".png")
        if p.exists(): return p
    raise FileNotFoundError(stem)

def _score_image(png_path, scaler, pca, clf):
    x = _load_image(png_path).flatten().reshape(1, -1)
    return float(clf.decision_function(pca.transform(scaler.transform(x)))[0])

print('Training functions defined')

Training functions defined


## 3. Cross-validation evaluation

In [4]:
augment_types = ['+All', '+AdvRot', '+AdvTrans', '+AdvBoth']
results = {}

for aug_type in augment_types:
    print(f"\n=== {aug_type} ===")
    fold_eers = []
    oof_scores = np.full(len(manifest), np.nan)
    
    for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
        seed_f = SEED + fold_id
        train_df = manifest.loc[train_idx]
        val_df = manifest.loc[val_idx]
        
        scaler, pca, clf = _train_image_with_aug(train_df, DATA, aug_type, seed_f)
        
        for idx, row in val_df.iterrows():
            oof_scores[idx] = _score_image(_find_png(row["stem"], DATA), scaler, pca, clf)
        
        # Fold EER
        scores_target = oof_scores[val_idx][y_all[val_idx] == 1]
        scores_nontarget = oof_scores[val_idx][y_all[val_idx] == 0]
        eer, _ = compute_eer(scores_target, scores_nontarget)
        fold_eers.append(eer * 100)
        print(f"  Fold {fold_id}: EER={eer*100:.2f}%")
    
    results[aug_type] = {
        'fold_eers': fold_eers,
        'mean': np.mean(fold_eers),
        'std': np.std(fold_eers),
        'oof_scores': oof_scores.copy()
    }
    print(f"  Mean ± Std: {np.mean(fold_eers):.2f} ± {np.std(fold_eers):.2f}%")

print("\n=== Summary ===")
for aug_type, res in results.items():
    print(f"{aug_type:12s}: {res['mean']:5.2f} ± {res['std']:5.2f}%  (fold_eers={res['fold_eers']})")


=== +All ===


  Fold 0: EER=2.08%


  Fold 1: EER=1.67%
  Fold 2: EER=0.83%
  Mean ± Std: 1.53 ± 0.52%

=== +AdvRot ===


  Fold 0: EER=0.69%


  Fold 1: EER=0.83%


...

  Fold 1: EER=1.67%


  Fold 2: EER=0.83%
  Mean ± Std: 3.43 ± 3.10%

=== +AdvBoth ===


  Fold 0: EER=1.39%


  Fold 1: EER=1.67%


  Fold 2: EER=8.33%
  Mean ± Std: 3.80 ± 3.21%

=== Summary ===
+All        :  1.53 ±  0.52%  (fold_eers=[2.083333333333333, 1.6666666666666667, 0.8333333333333334])
+AdvRot     :  0.51 ±  0.36%  (fold_eers=[0.6944444444444444, 0.8333333333333334, 0.0])
+AdvTrans   :  3.43 ±  3.10%  (fold_eers=[7.777777777777777, 1.6666666666666667, 0.8333333333333334])
+AdvBoth    :  3.80 ±  3.21%  (fold_eers=[1.3888888888888888, 1.6666666666666667, 8.333333333333332])


## 4. Stress test: evaluate on geometric degradations

In [5]:
def apply_stress(png_path, stress_type):
    """Apply stress to image and return stressed image."""
    x = _load_image(png_path)
    
    if stress_type == 'rot15':
        return _aug_rotate(x, 15)
    elif stress_type == 'rot25':
        return _aug_rotate(x, 25)
    elif stress_type == 'occlude':
        x_out = x.copy()
        x_out[30:50, 30:50] = 0  # 20x20 black square
        return x_out
    else:
        return x

print("\n=== Stress Test ===")
stress_types = ['clean', 'rot15', 'rot25', 'occlude']

for aug_type, res in results.items():
    print(f"\n{aug_type}:")
    
    # Train on full data
    scaler, pca, clf = _train_image_with_aug(manifest, DATA, aug_type, SEED)
    
    for stress in stress_types:
        scores_target = []
        scores_nontarget = []
        
        for idx, row in manifest.iterrows():
            png_path = _find_png(row["stem"], DATA)
            x_stressed = apply_stress(png_path, stress)
            score = _score_image(Path(str(png_path)), scaler, pca, clf)  # hack: score uses original path
            # Recompute with stressed image
            x_flat = x_stressed.flatten().reshape(1, -1)
            x_pca = pca.transform(scaler.transform(x_flat))
            score = float(clf.decision_function(x_pca)[0])
            
            if row["label"] == 1:
                scores_target.append(score)
            else:
                scores_nontarget.append(score)
        
        eer, _ = compute_eer(np.array(scores_target), np.array(scores_nontarget))
        print(f"  {stress:8s}: EER={eer*100:5.2f}%")


=== Stress Test ===

+All:


  clean   : EER= 0.00%
  rot15   : EER=13.70%


  rot25   : EER=34.11%
  occlude : EER= 0.00%

+AdvRot:


  clean   : EER= 0.00%
  rot15   : EER= 1.04%


  rot25   : EER=23.12%
  occlude : EER= 0.00%

+AdvTrans:


  clean   : EER= 0.00%
  rot15   : EER= 8.28%


  rot25   : EER=36.56%
  occlude : EER= 0.00%

+AdvBoth:


  clean   : EER= 0.00%
  rot15   : EER= 6.72%


  rot25   : EER=29.58%
  occlude : EER= 0.00%


## 5. Conclusion

Results:
- Clean EER: [TBD]
- Best adversarial config: [TBD]
- Robustness improvement: [TBD]

Decision: [Adopt/Reject]